# День 08. Упражнение 04
# Регрессия

## 0. Импорты

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import (
    KFold,
    cross_val_score,
    train_test_split,
    cross_validate,
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

## 1. Предобработка

1. Прочитай файл [`checker_regression.csv`](https://drive.google.com/file/d/1L8auBzJEghWFewznhhFpcrqxW4GqcfRY/view?usp=sharing).

In [2]:
df = pd.read_csv("../data/checker_regression.csv")
df.head()

,uid,num_commits,pageviews,AVG(diff)
0,user_1,62,28.0,NaN
1,user_1,62,28.0,NaN
2,user_1,62,28.0,NaN
3,user_1,62,28.0,NaN
4,user_1,62,28.0,NaN


2. Заполни пропущенные значения нулями (`0`).

In [4]:
df = df.fillna(0)
df.head()

,uid,num_commits,pageviews,AVG(diff)
0,user_1,62,28.0,0.0
1,user_1,62,28.0,0.0
2,user_1,62,28.0,0.0
3,user_1,62,28.0,0.0
4,user_1,62,28.0,0.0


In [5]:
df.value_counts().head(3)

uid      num_commits  pageviews  AVG(diff)
user_1   62           28.0       0.0          6
user_10  20           89.0       0.0          6
user_17  51           47.0       0.0          6
Name: count, dtype: int64

In [6]:
df = df.drop_duplicates().set_index("uid")
df.info()

<class 'pandas.DataFrame'>
Index: 29 entries, user_1 to user_8
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   num_commits  29 non-null     int64  
 1   pageviews    29 non-null     float64
 2   AVG(diff)    29 non-null     float64
dtypes: float64(2), int64(1)
memory usage: 928.0+ bytes


3. Раздели набор данных на обучающую и тестовую выборки с параметрами `test_size=0.2`.

In [7]:
y = df["AVG(diff)"]
X = df.drop(columns=["AVG(diff)"])

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21
)

## 2. Кросс-валидация

1. Напиши функцию `crossval`, которая принимает аргументы: `n_splits` для `KFold()`, `X`, `y`, модель (инициализированный класс с параметрами модели; не забудь о `random_state=21`) и возвращает для данного класса модели результат в таком виде:

```
train - 2696.4496895424836 | test - 1589.9979527104958
train - 2660.957874001452 | test - 2075.102636027137
train - 2847.315529246795 | test - 320.911928168403
train - 2500.7691099659237 | test - 4132.461382030178
train - 2643.927917295123 | test - 2237.8140952197878
train - 2396.295678819444 | test - 4509.650064742476
train - 2003.402267924976 | test - 8403.491474908551
train - 2531.876094212613 | test - 3135.944102735099
train - 2683.1795186023123 | test - 1796.01426292594
train - 2537.1192483996338 | test - 3439.29824116941
Average RMSE on crossval is 3164.0686140637476
```


In [10]:
def crossval(n_splits, X, y, model):
    """returns average neg_root_mean_squared_error (RMSE)"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=21)
    # cross_val_score не подходит т.к сохр-т только метрику для внутритестового фолда каждой итерации а то, как модель отработала на на своей внутренней обучающей выборке, функция cross_val_score считает и далее стирает из памяти.
    cv_results = cross_validate(
        model,
        X,
        y,
        cv=kf,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
        n_jobs=-1,
    )
    # Результаты работы функции cross_validate хранятся в обычном Python-словаре под ключами 'train_score' 'test_score'
    train_rmse = -cv_results["train_score"]
    test_rmse = -cv_results["test_score"]

    for train, test in zip(train_rmse, test_rmse):
        print(f"train - {train:<18} | test - {test}")

    mean_test_rmse = np.mean(test_rmse)
    print(f"Average RMSE on crossval is {mean_test_rmse}")
    return mean_test_rmse

2. Запусти функцию для моделей `LinearRegression`, `DecisionTreeRegressor`, `RandomForestRegressor`. Параметры можешь выбрать самостоятельно, но тщательно убедись, что они подходят.

In [11]:
print("результаты моделей LinearRegression:")
print("-" * 40)
for n_splits in [5, 10, 15]:
    print(f"max_depth={n_splits}")
    crossval(n_splits, X_train, y_train, LinearRegression())
    print("-" * 60)

результаты моделей LinearRegression:
----------------------------------------
max_depth=5


train - 48.9650182985128   | test - 104.57142340670634
train - 67.12604324415598  | test - 61.64346142808743
train - 66.61303918454013  | test - 51.37782993510151
train - 62.66428544831268  | test - 65.76577742150987
train - 64.32380789393129  | test - 60.75794925163608
Average RMSE on crossval is 68.82328828860825
------------------------------------------------------------
max_depth=10
train - 53.33351096491692  | test - 115.2394318363867
train - 62.409919772255556 | test - 69.35808794733022
train - 64.75146423244131  | test - 52.395419621714424
train - 64.04094981627755  | test - 78.18810821407222
train - 63.574089717062186 | test - 58.669554693832154
train - 65.40154560181439  | test - 32.10068854674122
train - 64.40600245557161  | test - 48.59322878192214
train - 61.215472663804725 | test - 82.41734194163652
train - 64.69372263304507  | test - 43.81163345544473
train - 62.80195740062014  | test - 70.47228900535454
Average RMSE on crossval is 65.1245784044435
----------------------

LinearRegression: С ростом числа фолдов средняя ошибка Average RMSE снизилась с 68.82 до 63.95 за счет увеличения объема обучающей выборки на каждом шаге. Оптимальным выбором считаю n_splits=10

In [12]:
print("результаты моделей DecisionTree:")
print("-" * 40)
for depth in [4, 6, 8]:
    print(f"max_depth={depth}")
    model_dt = DecisionTreeRegressor(max_depth=depth, random_state=21)
    crossval(10, X_train, y_train, model_dt)
    print("-" * 60)

результаты моделей DecisionTree:
----------------------------------------
max_depth=4
train - 19.756827933653724 | test - 63.2101257711136
train - 20.876122724299165 | test - 82.25810891019488
train - 22.063461184602122 | test - 11.518137870333035
train - 21.742264634314694 | test - 8.60313250443633
train - 15.222075077495283 | test - 80.25661966467314
train - 21.569529200867827 | test - 15.202795795510772
train - 21.347979223599054 | test - 40.58792924010783
train - 21.569529200867827 | test - 37.190388543278225
train - 20.39538654519797  | test - 30.8121779902038
train - 13.121372766613382 | test - 69.11968786098502
Average RMSE on crossval is 43.87591041508366
------------------------------------------------------------
max_depth=6
train - 0.0                | test - 43.200154320712016
train - 12.965338406690355 | test - 83.15046156973068
train - 0.0                | test - 9.093266739736606
train - 0.0                | test - 8.60313250443633
train - 0.0                | test - 80.

DecisionTree: train - 0.0!!  при max_depth=6 
Дерево решений полностью заучило тренировочный датасет.

In [15]:
results = []

print("результаты моделей RandomForest:")
for estimators in range(100, 250, 50):
    for depth in range(5, 20, 5):
        # print(f"estimators={estimators}, max_depth={depth}")
        model_rf = RandomForestRegressor(
            n_estimators=estimators, max_depth=depth, random_state=21, n_jobs=-1
        )
        AverageRMSE = crossval(10, X_train, y_train, model_rf)
        results.append(
            {
                "estimators": estimators,
                "max_depth": depth,
                "train_rmse": AverageRMSE,
            }
        )
        print("-" * 60)

результаты моделей RandomForest:
train - 20.63085190341936  | test - 76.83328744586956
train - 23.19186326431637  | test - 81.2838793557124
train - 23.063995847128204 | test - 5.501420331756774
train - 23.72053292111506  | test - 11.17332878618697
train - 24.802698396786475 | test - 69.06849984182773
train - 20.918185796239708 | test - 39.78281186700455
train - 21.64642166645181  | test - 55.97960571048663
train - 23.319924381128516 | test - 24.10209705184025
train - 20.853088203556823 | test - 26.21383867001997
train - 17.113184803676937 | test - 69.6015772708772
Average RMSE on crossval is 45.9540346331582
------------------------------------------------------------
train - 19.85306588532919  | test - 76.53685427174808
train - 22.671373358544045 | test - 81.61793883314117
train - 21.694829255044976 | test - 5.71235319286195
train - 22.7272784086383   | test - 9.177142450433896
train - 24.38477655695876  | test - 70.14702143355767
train - 19.90791216218836  | test - 40.07320878716852


In [ ]:
for model_characteristics in results:
    print(f"model: {model_characteristics}")

model: {'estimators': 100, 'max_depth': 5, 'train_rmse': np.float64(45.9540346331582)}
model: {'estimators': 100, 'max_depth': 10, 'train_rmse': np.float64(45.759900048221446)}
model: {'estimators': 100, 'max_depth': 15, 'train_rmse': np.float64(45.759900048221446)}
model: {'estimators': 150, 'max_depth': 5, 'train_rmse': np.float64(46.28030868394969)}
model: {'estimators': 150, 'max_depth': 10, 'train_rmse': np.float64(46.150702241205146)}
model: {'estimators': 150, 'max_depth': 15, 'train_rmse': np.float64(46.15070224120514)}
model: {'estimators': 200, 'max_depth': 5, 'train_rmse': np.float64(45.51710915111661)}
model: {'estimators': 200, 'max_depth': 10, 'train_rmse': np.float64(45.38257040437203)}
model: {'estimators': 200, 'max_depth': 15, 'train_rmse': np.float64(45.38257040437203)}


Лучший результат у модели RandomForest с пар-ми: estimators=200, max_depth=10 с ошибкой 45.3825.

## 3. Предсказания и оценка

1. Сделай предсказания на тестовом наборе данных для каждой из трёх моделей с финальными параметрами.
2. Построй график для каждой модели, где по оси x — фактическая средняя разница, а по оси y — предсказание модели.
3. Как должен выглядеть график в идеальном случае? Напиши ответ в markdown-ячейке в конце раздела.